# RSNA Knee — Error Report

CPU-only kernel (zero GPU quota; runs alongside training): scores the latest
train-kernel checkpoint over all 58 gold studies plus a sample, and renders the
error-analysis HTML — the gold audit (model-caught-label-error vs model-error
buckets that pick the next lever), model/miner/gold AUC tables, and per-study
panels with per-label attention. Mounts the train kernel's latest completed
output via kernel_sources, so rerunning this after any training run audits that
run's checkpoint. The HTML lands in this kernel's output (private; contains
StudyInstanceUIDs — internal use only, never republish).

In [ ]:
# Pin a commit so the report traces to exact code.
COMMIT = "0970afb"  # main: report decodes in the checkpoint's laterality frame
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

In [ ]:
from pathlib import Path

from knee.report import build_error_report

SLUG = "rsna-knee-abnormality-detection"
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next(p for p in candidates if (p / "train.csv").exists())


def find_input(name: str, pattern: str) -> Path:
    bases = [
        Path("/kaggle/input") / name,
        Path("/kaggle/input/datasets/josiemachalek") / name,
        Path("/kaggle/input/notebooks/josiemachalek") / name,  # kernel_sources mount here
    ]
    for base in bases:
        matches = sorted(base.rglob(pattern)) if base.exists() else []
        if matches:
            return matches[0]
    listing = {str(d): [x.name for x in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{name}/{pattern} not found; mounts: {listing}")


labels_csv = find_input("knee-labels", "blended_labels_v1.csv")
checkpoint = find_input("rsna-knee-train", "*_unified.pt")  # latest train run's multiplane checkpoint
print("auditing:", checkpoint.name)

build_error_report(
    COMP_ROOT, labels_csv, checkpoint, Path("/kaggle/working/error_report.html"), sample=120
)

In [ ]:
# Print the aggregate tables so the verdict is readable from the log alone.
import re

page = Path("/kaggle/working/error_report.html").read_text()
buckets = dict(re.findall(r"<tr><td>(both right|model caught label error|model error \(labels fine\)|both wrong)</td><td>(\d+)</td></tr>", page))
print("gold audit buckets:", buckets)
for match in re.finditer(r"<h3>([^<]+\(macro [^)]+\))</h3>", page):
    print(match.group(1))